# 🎥 ChatTube: Conversational RAG with YouTube Transcripts

Welcome to **ChatTube**! This notebook walks you through building a **Retrieval-Augmented Generation (RAG)** pipeline to have a natural, contextual conversation with any YouTube video.

### What We Will Cover:
1. **Setup & Imports**: Load environment variables and import LangChain components.
2. **Ingestion & Indexing**: Download raw transcripts, split them into semantic chunks, and build a local `FAISS` vector store using Google Gemini Embeddings.
3. **Retrieval**: Query the vector store to fetch relevant document context.
4. **Augmentation**: Construct system prompts and handle conversational memory.
5. **Conversational Chain (LCEL)**: Build a custom chain with query contextualization to rewrite follow-up questions.
6. **Live Chat Loop**: Run an interactive conversational loop directly in the notebook.

## Setup & Imports

In [1]:
import os
from operator import itemgetter
from dotenv import load_dotenv

# Transcript loader and Document wrapper
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_core.documents import Document

# Text splitting & chunking utility
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Gemini Embeddings & Vector Store
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Chat Model & Prompts
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# LCEL & Output parsing
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Load the API keys from your .env file
load_dotenv()

/Users/noamankhan/Desktop/Learning/GenAI/LangChain/LangChainPractice/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/7n/9ddnlj_93hd6z10bnfbvxtdm0000gn/T/ipykernel_54018/3583341349.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

## Indexing

### 1. Document Loading

We use the `youtube-transcript-api` library to download the raw transcript for our target YouTube video. 

Then we merge the transcript snippets into a single contiguous block of text and wrap it in a LangChain `Document` object.

In [2]:
# Video ID of interest
video_id = "NVwTN0OxgKQ"

# Initialize the YouTube Transcript API client
ytt_api = YouTubeTranscriptApi()

# Safely attempt to fetch the transcript
try:
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    print("Successfully fetched transcript snippets.")
    # Print a preview of the first few snippets
    for i, snippet in enumerate(fetched_transcript[:5]):
        print(f"Snippet {i}: {snippet}")
except TranscriptsDisabled:
    print(f"Transcripts are disabled for video ID: {video_id}")
    fetched_transcript = None
except Exception as e:
    print(f"Failed to retrieve transcript for video ID {video_id}: {e}")
    fetched_transcript = None

Successfully fetched transcript snippets.
Snippet 0: FetchedTranscriptSnippet(text='At the start of each season, I I tend to', start=7.759, duration=5.92)
Snippet 1: FetchedTranscriptSnippet(text='address the group.', start=10.719, duration=5.201)
Snippet 2: FetchedTranscriptSnippet(text="I think it's the right thing for one of", start=13.679, duration=5.68)
Snippet 3: FetchedTranscriptSnippet(text='the leaders to do.', start=15.92, duration=6.48)
Snippet 4: FetchedTranscriptSnippet(text='At the start of this season, we did use', start=19.359, duration=7.521)


In [3]:
# Extract the text and join them only if fetching succeeded
if fetched_transcript:
    transcript = " ".join([snippet.text for snippet in fetched_transcript])
else:
    transcript = ""
    print("No transcript data available to join.")

In [4]:
transcript

'At the start of each season, I I tend to address the group. I think it\'s the right thing for one of the leaders to do. At the start of this season, we did use a quote um that was change is the only constant. And the reason I chose that quote was because firstly we had a philosophy of um not trying to defend our title rather hunt the hunt the next one. One of the dangers in a human condition is that we try to hold on to stuff that\'s good and that\'s I think an unhealthy emotional place to be. Understanding that everything is changing all the time, I think is a very healthy place to be. I didn\'t want us to try to hold on to the fact that we played a certain style last year or had a certain success last year. This year was different and I think those are good life lessons for our players. For me personally, I have an equation with a lot of the batters. >> Beautiful. >> Different players uh are in different parts of the world and doing different things. I\'ll be everything being a comm

In [5]:
# Wrap the transcript in a LangChain Document if we have text
if transcript:
    document = Document(page_content=transcript, metadata={"video_id": video_id})
    print("Document wrapped successfully.")
else:
    document = None
    print("Cannot create Document: transcript is empty.")

Document wrapped successfully.


### 2. Text Splitting

Because LLMs have context window limits, and we want to retrieve precise segments of the video, we split the long transcript document into smaller, overlapping chunks using `RecursiveCharacterTextSplitter`.

In [6]:
# Configure splitter to chunk text by paragraphs/sentences/words
# Chunk size: 500 characters, overlap: 100 characters to preserve context boundary
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". ", " ", ""],
    chunk_size=500, 
    chunk_overlap=100
)

In [7]:
# Split document into chunks only if the document was successfully created
if document:
    chunks = text_splitter.split_documents([document])
    print(f"Split transcript into {len(chunks)} document chunks.")
else:
    chunks = []
    print("Cannot split documents: no document available.")

Split transcript into 33 document chunks.


### 3. Embedding Generation and Vector Store

We convert the text chunks into vector embeddings using Google GenAI's embedding model. Then, we index these embeddings into a local `FAISS` vector store, which allows us to perform fast similarity searches later.

In [8]:
# Initialize the Gemini Embeddings client
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", task_type="SEMANTIC_SIMILARITY")

In [9]:
# Create a local FAISS vector store index from the document chunks using our embeddings
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store index initialized and populated successfully!")

Vector store index initialized and populated successfully!


In [10]:
# Print the mapping of index positions to internal document store IDs
vector_store.index_to_docstore_id

{0: '948532d1-451d-42a3-9d4a-68f9aec92619',
 1: '13228923-6d09-4997-863a-01ca9929bfd4',
 2: 'd84c51a1-27df-40d1-8441-a8117d18f561',
 3: 'e84c8587-e794-4b36-b1d7-47448c043146',
 4: 'b7b06520-2378-4194-adef-9caf252e6bd2',
 5: '169e9093-15c0-45eb-87e5-b41ef1a32bb8',
 6: '8e1bcceb-c973-4277-ae08-7ff399dff19d',
 7: 'e5039b00-61d7-4e0a-a90e-d1c89f677d2d',
 8: 'a7e17630-d0df-4c4c-b975-571cb2bf7aab',
 9: '8b3c25b6-522c-4b61-aceb-3dc916c91566',
 10: '89ef311a-113b-42c8-94e0-1020e4fd7701',
 11: '5f0db39e-3f80-4590-8680-9f371961ee3d',
 12: 'b219a81b-c5d8-405d-b9fc-2577d74b9543',
 13: 'e4e44f37-2ce5-4751-88f0-9c14c2dcca87',
 14: 'c730f7a3-1c4e-4345-88ab-67e7b63f178b',
 15: '8be4714b-fa90-4b7e-9b1c-fd376f16923f',
 16: '01083a04-8a48-49eb-8ba4-8d8af59d2c96',
 17: '6867bc98-abfa-4a27-a9a4-a8157ae27e65',
 18: '8a026378-0094-4213-8133-b3a03cad8d4c',
 19: '8669fcab-3fdb-497c-b5fa-3b5406fc98c0',
 20: '493f4b28-c61c-4b52-9faa-de5bd2575f14',
 21: 'd8063c74-935e-48d7-8845-3a8e555e42fb',
 22: '259263f0-49ae-

In [11]:
# Verify retrieval of a specific chunk by its document ID
# (Make sure to replace this ID with one from the list printed above if it raises an error)
sample_id = list(vector_store.index_to_docstore_id.values())[0]
vector_store.get_by_ids([sample_id])

[Document(id='948532d1-451d-42a3-9d4a-68f9aec92619', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content="At the start of each season, I I tend to address the group. I think it's the right thing for one of the leaders to do. At the start of this season, we did use a quote um that was change is the only constant. And the reason I chose that quote was because firstly we had a philosophy of um not trying to defend our title rather hunt the hunt the next one. One of the dangers in a human condition is that we try to hold on to stuff that's good and that's I think an unhealthy emotional place to be")]

## 2. Retrieval

We define our retriever component. Given a user question, the retriever queries the vector store, computes vector similarity, and returns the top 5 document chunks containing the relevant video context.

In [12]:
# Instantiate the retriever using a similarity search strategy (fetching top 5 chunks)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [13]:
# Perform a test retrieval for a general video query
retriever.invoke("What is this video about?")

[Document(id='297eb566-872b-4632-bb1a-cc222f945899', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". As the head coach, you're expected to say something in the huddle at the start of each game. It's quite tricky to come up with something to say each game, something that will keep players heading in the right direction. You don't want to say things in that huddle that is a waste of time. So, you need to pick what you want to say quite carefully. You need to change it up quite often"),
 Document(id='54d3b218-ec36-4d15-b9c2-33885a05c677', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". So, you need to pick what you want to say quite carefully. You need to change it up quite often. It needs to be sufficiently succinct that you're not bombarding them with information just before they're going into pressure situations. They need clear minds and clear heads. But it also needs to justify the fact that you're bringing them into a huddle"),
 Document(id='8a026378-0094-4213-8133-b3a03

In [14]:
# Perform a test retrieval for a summarization query
retriever.invoke("Can you summarize the main points?")

[Document(id='54d3b218-ec36-4d15-b9c2-33885a05c677', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". So, you need to pick what you want to say quite carefully. You need to change it up quite often. It needs to be sufficiently succinct that you're not bombarding them with information just before they're going into pressure situations. They need clear minds and clear heads. But it also needs to justify the fact that you're bringing them into a huddle"),
 Document(id='297eb566-872b-4632-bb1a-cc222f945899', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". As the head coach, you're expected to say something in the huddle at the start of each game. It's quite tricky to come up with something to say each game, something that will keep players heading in the right direction. You don't want to say things in that huddle that is a waste of time. So, you need to pick what you want to say quite carefully. You need to change it up quite often"),
 Document(id='8b3c25b6-522c-4b61-aceb-3dc91

## 3. Augmentation

Here we initialize the Google Gemini Chat Model and set up the system instruction. The system instruction prompts the LLM to act as "ChatTube" and strictly rely on the retrieved transcript context to answer queries.

In [33]:
# Initialize the Chat Model (Gemini 2.5 Flash) with moderate temperature for balanced responses
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3)

In [16]:
# System instruction guiding the model to use context and handle missing information honestly
system_prompt = (
    "You are ChatTube, a specialized AI assistant that answers questions based on a YouTube video's transcript.\n"
    "Use the retrieved video segments below to answer the user's question. "
    "If you don't know the answer or if the information is not in the transcript, "
    "honestly state that you don't know based on the video content.\n\n"
    "Video Transcript Context:\n"
    "{context}"
)

In [17]:
# Create a ChatPromptTemplate with System message, chat history placeholder, and user input
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [18]:
# Retrieve the context documents for our test question
retrieved_docs = retriever.invoke("What is this video about?")

In [19]:
retrieved_docs

[Document(id='297eb566-872b-4632-bb1a-cc222f945899', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". As the head coach, you're expected to say something in the huddle at the start of each game. It's quite tricky to come up with something to say each game, something that will keep players heading in the right direction. You don't want to say things in that huddle that is a waste of time. So, you need to pick what you want to say quite carefully. You need to change it up quite often"),
 Document(id='54d3b218-ec36-4d15-b9c2-33885a05c677', metadata={'video_id': 'NVwTN0OxgKQ'}, page_content=". So, you need to pick what you want to say quite carefully. You need to change it up quite often. It needs to be sufficiently succinct that you're not bombarding them with information just before they're going into pressure situations. They need clear minds and clear heads. But it also needs to justify the fact that you're bringing them into a huddle"),
 Document(id='8a026378-0094-4213-8133-b3a03

In [20]:
# Combine the page content of the retrieved documents into a single formatted string
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [21]:
context_text

". As the head coach, you're expected to say something in the huddle at the start of each game. It's quite tricky to come up with something to say each game, something that will keep players heading in the right direction. You don't want to say things in that huddle that is a waste of time. So, you need to pick what you want to say quite carefully. You need to change it up quite often\n\n. So, you need to pick what you want to say quite carefully. You need to change it up quite often. It needs to be sufficiently succinct that you're not bombarding them with information just before they're going into pressure situations. They need clear minds and clear heads. But it also needs to justify the fact that you're bringing them into a huddle\n\n. >> So our dugout is uh very consistent with where we sit more at the top right corner you can see actually and then it's Andy and then it's Malo and then it's me next to me is Freddy and then Omar and then towards the back end it's stick. >> Andy and

In [22]:
# Format the template into a ChatPromptValue by passing the context, question, and an empty history
final_prompt = qa_prompt.invoke({"context": context_text, "input": "What is this video about?", "chat_history": []})

## 4. Generation

We pass the structured, context-augmented prompt to the model to generate the final response.

In [25]:
# Invoke the LLM with the formatted prompt
answer = model.invoke(final_prompt)

In [26]:
# Extract and print the generated text content
answer.content

'Based on the provided transcript segments, this video appears to be about coaching strategies and team management, focusing on:\n\n*   **Huddle Communication:** The challenges and importance of a head coach\'s speech in the huddle at the start of each game, emphasizing the need for careful, succinct, and varied messages to keep players focused without overwhelming them.\n*   **Dugout Environment:** How coaches position themselves in the dugout (e.g., hiding in a corner) to create a "players\' environment" and their philosophy behind this setup.\n*   **Player Management:** The importance of understanding and meeting the diverse experiences, styles, and preferences of individual players, noting different approaches to challenges among players like Bouvenes Akuma and Hazelwood.'

## 5. Building the Chain (with Query Contextualization)

To build our RAG pipeline, we construct a custom chain using LangChain Expression Language (LCEL).

**Key Improvement**: We add a query contextualizer sub-chain. This sub-chain inspects the user question and the chat history. If the question refers back to previous topics, it rewrites the question into a standalone query. This standalone query is then passed to the retriever.

In [34]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [35]:
str_parser = StrOutputParser()

In [36]:
# Prompt to contextualize query based on history
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Sub-chain that rewrites follow-up questions
query_contextualizer = contextualize_q_prompt | model | str_parser

# Custom LCEL Chain utilizing the contextualizer sub-chain for Retrieval
parallel_chain = RunnableParallel({
    "input": itemgetter("input"),
    "chat_history": itemgetter("chat_history"),
    # The query is first contextualized, then retrieved, then formatted
    "context": query_contextualizer | retriever | RunnableLambda(format_docs)
})


In [37]:
rag_chain = parallel_chain | qa_prompt | model | str_parser

In [38]:
rag_chain.invoke({"input" : "What is this video about?","chat_history":[]})

'Based on the transcript provided, this video is about the team dynamics, leadership, and communication strategies within a cricket team (likely an IPL team).\n\nKey themes include:\n\n*   **Virat Kohli\'s Leadership:** The speakers highlight Kohli’s confidence and his ability to deliver "inch-perfect" speeches during team huddles. He is noted for his composure, such as his assurance to teammates during high-pressure chases.\n*   **Huddle Communication:** The video discusses the importance of keeping pre-game huddle talks succinct and impactful to ensure players have "clear minds and clear heads" before entering pressure situations.\n*   **Player Management:** It touches on the challenge of staying in touch with players who are scattered across different parts of the world and ensuring the team remains prepared for the increased difficulty of upcoming seasons.\n*   **Team Preparation:** The speakers emphasize the need for continuous hard work and staying ahead of the competition, notin

## 6. Chat with the Video

We wrap our finalized RAG chain inside an interactive terminal loop. We maintain a list of `HumanMessage` and `AIMessage` objects in `chat_history` and append new turns on each interaction to retain contextual memory.

In [39]:
def ask_question(question):
    global chat_history
    
    # Invoke the RAG chain passing the current question and history
    try:
        response = rag_chain.invoke({
            "input": question,
            "chat_history": chat_history
        })
        
        print(f"🤖 AI: {response}\n")
        
        # Append both messages to the history list to maintain chat context
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=response))
    except Exception as e:
        # Handle temporary 503 Service Unavailable / High Demand spikes
        if "503" in str(e) or "UNAVAILABLE" in str(e).upper() or "high demand" in str(e).lower():
            print("🤖 AI: The server is currently experiencing high demand. Please try again in a few seconds.\n")
        else:
            print(f"🤖 AI: An error occurred while generating a response: {e}\n")

In [40]:
# Initialize the chat history store
chat_history = []

print("ChatTube Live Chat started! Type \'exit\', \'bye\', or \'close\' to finish.\n")

# Loop indefinitely to allow back-and-forth conversation
while True:
    user_query = input("👤 User: ")
    
    # Check for exit commands
    if user_query.lower().strip() in ["exit", "bye", "close"]:
        print("🤖 AI: Goodbye!")
        break
        
    # Ignore empty queries
    if not user_query.strip():
        continue
        
    ask_question(user_query)

ChatTube Live Chat started! Type 'exit', 'bye', or 'close' to finish.



👤 User:  Did virat kohli said anything in the video ?


🤖 AI: Yes, the video mentions specific things Virat Kohli said during matches:

*   **During the final:** He told the team, "Just tell the others to bat, I'm going to be here, not out. Just tell them to do their thing."
*   **During the KKR game:** While batting, he said, "Don't worry, DK will get this job done."

Additionally, the transcript notes that Virat Kohli is "outstanding" when speaking in team huddles and "almost always gets it inch perfect" when he addresses the team.



👤 User:  summarise the video.


🤖 AI: The video provides an inside look at the team culture, leadership, and competitive mindset within the Royal Challengers Bengaluru (RCB) dugout. The key points are:

*   **Virat Kohli’s Leadership and Confidence:** Kohli is highlighted for his immense self-belief and ability to inspire teammates. Whether it is telling his team he will stay "not out" to anchor a chase or reassuring them that a teammate (like DK) will get the job done, his presence is described as a major source of confidence for the squad.
*   **The Value of Team Huddles:** The coaching staff emphasizes that team huddles are carefully planned to provide meaningful input and variation. Virat Kohli is specifically praised for his ability to deliver "inch-perfect" messages during these moments.
*   **Competitive Mindset:** The video notes that champions like Kohli thrive in high-pressure situations and actually "smile at the challenge" rather than shying away from it.
*   **Strategic Adaptation:** The team made a cons

👤 User:  tell about the dugout of RCB team.


🤖 AI: Based on the transcript, the RCB dugout is described as a "fascinating" place that features a unique mix of different personalities and styles. Here are the key details provided about it:

*   **Diverse Dynamics:** It is characterized as a real blend of various personalities, which makes the environment interesting.
*   **Roles and Responsibilities:** The team actively utilizes the experience and wisdom of specific players. For instance, **DK (Dinesh Karthik)** is noted as being the most active person in the dugout, a role and responsibility that the team has specifically assigned to him to leverage his intuition and experience.
*   **Energy and Perspective:** The speaker (the head coach) describes himself as the most "excited" person in the dugout, noting that he watches the game through a different lens than others.
*   **Calm and Consistent Approach:** The dugout strives to remain relatively calm. They make a conscious effort not to "ride the wins and the losses too much," bel

👤 User:  exit


🤖 AI: Goodbye!
